# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [4]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," which is mentioned multiple times in the project list.'

In [11]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, one project titled "Pathfinder 24" in the Healthcare / MedTech domain with a secondary focus on Security. The description indicates it is an "AI-powered platform optimizing logistics routes for sustainability," but given the secondary domain, security considerations are likely addressed within that context.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had various comments about the fintech projects, highlighting strengths and areas for improvement. For example, they described some projects as a "clever solution with measurable environmental benefit" and noted that others were "technically ambitious and well-executed." Some judges appreciated the "impressive real-world impact" and "robust experimental validation," while noting that certain projects could benefit from additional benchmarking or qualitative analysis. Overall, the judges recognized the quality and potential of the fintech projects, with positive remarks emphasizing their innovation, quality, and impact.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain cannot be determined definitively since only a small sample of projects is shown. However, among the listed projects, the domains represented are Productivity Assistants, Legal / Compliance, Data / Analytics, and Healthcare / MedTech. \n\nIf you have access to the full dataset, I recommend counting the occurrences of each domain to identify the most common one. \n\nWould you like assistance with that?'

In [16]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no explicit mentions of use cases related to security.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges described the fintech project (specifically the "PulseAI 50" in the Finance / FinTech secondary domain) as "Technically ambitious and well-executed."'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer

Example: "Which projects have 'AI' in their title?"

BM25 would likely outperform embeddings because:
- Since the example specifically targets the"title, BM25's ability to match exact terms becomes crucial. Embeddings might return projects that are about artificial intelligence concepts but have titles like "Machine Learning Platform" or "Intelligent System" without the exact term "AI".
- This is a precise lookup query rather than a conceptual question. The user wants documents that literally contain "AI" in the title, not documents that are conceptually about AI.
- Acts more like a traditional database search for exact matches. 

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Productivity Assistants," followed by other domains such as "Creative / Design / Media" and "Security." The dataset snippet indicates that "Productivity Assistants" is listed as a project domain multiple times, suggesting it may be the most prevalent.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases explicitly related to security. The projects mentioned focus on federated learning to improve privacy in healthcare applications, but there is no direct mention of security-related use cases.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had positive comments about the fintech projects. Specifically, for the project "PlanPilot," which is in the Finance / FinTech domain, the judges noted it as "a clever solution with measurable environmental benefit" and gave it a high score of 8.4.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Healthcare / MedTech," which is mentioned multiple times across different projects.'

In [26]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, one project titled "InsightAI 1" operates in the Security domain with a focus on a "low-latency inference system for multimodal agents in autonomous systems." Additionally, another project called "SecureNest 12" is in the Security domain, which involves a "low-latency inference system for multimodal agents in autonomous systems."'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges generally had positive comments about the fintech projects. They described some projects as clever solutions with measurable environmental benefits, comprehensive and technically mature approaches, well-structured and scalable with good potential for commercialization, and impressive in real-world impact. For example, one judge called a project "a clever solution with measurable environmental benefit," while others noted that certain projects were "technically ambitious and well-executed," or "solid work with impressive real-world impact." Overall, judges appreciated the technical quality, innovation, and potential applications of the fintech projects.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer
It helps ensure that relevant information isn't missed due to  variations between the user's query and how information is expressed in the documents. For instance, if a user asks "Were there any usecases about security?", the multi-query retriever might generate additional queries to capture things like security related project examples, project involving data protection etc. 

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned multiple times in the sample provided.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases explicitly related to security mentioned. The projects listed focus on privacy improvements in healthcare applications through federated learning, but there is no direct mention of security use cases.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech projects. Specifically, for the project "PlanPilot 35," judges called it "a clever solution with measurable environmental benefit." Additionally, other projects received praise such as being "comprehensive and technically mature," "technically ambitious and well-executed," and "solid work with impressive real-world impact."'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Healthcare / MedTech," as it is mentioned multiple times across different projects.'

In [39]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided information, there are no specific usecases explicitly mentioned about security.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had varied comments about the fintech projects. For the project "Pathfinder 27," judges praised the excellent code quality and use of open-source libraries, giving it a high score of 9.8. Similarly, "PulseAI 50" was described as technically ambitious and well-executed, with a strong score of 8.0. Overall, the judges recognized the quality, technical execution, and potential impact of the fintech projects.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [43]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Legal / Compliance," which appears twice among the listed projects.'

In [47]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the project titled "BioForge" is a medical imaging solution categorized under the Security domain. Additionally, "Project Aurora" focuses on a low-latency inference system for multimodal agents in autonomous systems, also within the Security domain.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech projects, often highlighting their technical maturity, potential for commercialization, and strong data support. For example, one project was described as "Comprehensive and technically mature," while another was noted for being "Well-structured and scalable; good potential for commercialization." Overall, the judges recognized the fintech projects for their ambitious, well-executed approaches and potential impact.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer
For repetitive and short sentences semantic chunking may not be optimal because it  relies on calculating distances between sentence embeddings. With highly similar and repetitive sentences, these distances will be very small, making it difficult to find meaningful breakpoints. Additionally, FAQs are typically organized by topic or category, but semantic chunking might break this logical organization by grouping sentences based on structural similarity rather than content relevance. 

Re adjusting the algorithm: Rule-based or metadata-driven chunking strategies may work better for maintaining logical organization and ensuring each chunk contains complete, meaningful information.



# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

#### ✅ Answer

##### Generate Dataset

In [51]:
from langchain.evaluation import load_evaluator
from langchain.evaluation import EvaluatorType
from langchain.evaluation import QAEvalChain
from langchain_core.prompts import PromptTemplate
import time

# Hardcoded Synthetic Dataset for LangChain Evaluation
synthetic_dataset = [
    {
        "question": "What is the most common project domain?",
        "answer": "Based on the data, Developer Tools/DevEx, Security, and Finance/FinTech are the most common project domains with multiple projects in each category.",
        "contexts": []  # Will be populated by retrievers
    },
    {
        "question": "Which projects received the highest scores?",
        "answer": "The highest scoring projects include ChatBridge 9 (97), AutoMate 11 (97), MediMind 21 (97), Pathfinder 23 (97), and Pathfinder 24 (97).",
        "contexts": []
    },
    {
        "question": "What did judges say about fintech projects?",
        "answer": "Judges commented positively on fintech projects, noting 'technically ambitious and well-executed' approaches, 'solid work with impressive real-world impact', and 'well-structured and scalable' solutions.",
        "contexts": []
    },
    {
        "question": "Which projects use medical imaging?",
        "answer": "Medical imaging projects include WealthifyAI 3 (vision transformers for early diagnosis), MediMind 17 (medical imaging solution), and SkillMatch 31 (medical imaging with vision transformers).",
        "contexts": []
    },
    {
        "question": "What are the BioForge projects about?",
        "answer": "BioForge projects focus on bioinformatics pipelines, graph neural networks for industrial process optimization, and genome annotation using transformers.",
        "contexts": []
    },
    {
        "question": "Which projects scored above 90?",
        "answer": "Projects scoring above 90 include WealthifyAI 3 (91), TrendLens 6 (88), InsightAI 1 (85), SecureNest 18 (82), and several others with scores ranging from 82-97.",
        "contexts": []
    },
    {
        "question": "What feedback did judges give about UI design?",
        "answer": "Judge feedback on UI design includes comments like 'UI design feels rushed' for some projects, while others received praise for 'excellent code quality' and 'well-articulated research scope'.",
        "contexts": []
    },
    {
        "question": "Which projects focus on security applications?",
        "answer": "Security-focused projects include InsightAI 1 (multimodal agents), SecureNest 12 (low-latency inference), InsightAI 36 (synthetic data generation), and others focusing on security and compliance applications.",
        "contexts": []
    }
]

print(f"Created LangChain evaluation dataset with {len(synthetic_dataset)} questions")

Created LangChain evaluation dataset with 8 questions


##### Evaluation

In [52]:
qa_evaluator = load_evaluator("qa")
criteria_evaluator = load_evaluator("criteria", criteria="helpfulness")
context_evaluator = load_evaluator("context_qa")

# Custom evaluation prompt for retriever performance
retriever_eval_prompt = PromptTemplate(
    input_variables=["question", "retrieved_context", "expected_answer"],
    template="""
    Question: {question}
    
    Retrieved Context: {retrieved_context}
    
    Expected Answer: {expected_answer}
    
    Rate the quality of the retrieved context for answering the question on a scale of 1-5:
    1 = Poor (context doesn't help answer the question)
    2 = Below Average (context partially relevant)
    3 = Average (context somewhat relevant)
    4 = Good (context mostly relevant and helpful)
    5 = Excellent (context highly relevant and comprehensive)
    
    Score: """
)

# Create custom retriever evaluator
def create_retriever_evaluator():
    """Create a custom evaluator for retriever performance"""
    return retriever_eval_prompt | chat_model | StrOutputParser()

In [ ]:
def evaluate_retriever_langchain(retriever_name, retriever, rag_chain):
    """Evaluate retriever using LangChain evaluators"""
    
    print(f"\n🔍 Evaluating {retriever_name} with LangChain...")
    
    results = []
    start_time = time.time()
    
    for i, item in enumerate(synthetic_dataset):
        question = item['question']
        expected_answer = item['answer']
        
        print(f"  Q{i+1}: {question[:50]}...")
        
        try:
            # Get answer from RAG chain
            response = rag_chain.invoke({"question": question})
            generated_answer = response["response"].content
            
            # Get retrieved contexts
            retrieved_docs = retriever.get_relevant_documents(question)
            contexts = [doc.page_content for doc in retrieved_docs]
            context_text = "\n".join(contexts)
            
            # Evaluate answer quality using  QA evaluator
            qa_eval_result = qa_evaluator.evaluate_strings(
                prediction=generated_answer,
                reference=expected_answer,
                input=question
            )
            
            # Evaluate context relevance using custom evaluator
            context_eval_prompt = retriever_eval_prompt.format(
                question=question,
                retrieved_context=context_text[:500],  # Limit context length
                expected_answer=expected_answer
            )
            
            context_score_response = chat_model.invoke(context_eval_prompt)
            context_score = context_score_response.content.strip()
            
            # Extract numeric score from response
            try:
                context_score_num = float(context_score.split()[-1])
            except:
                context_score_num = 3.0  # Default score
            
            results.append({
                "question": question,
                "generated_answer": generated_answer,
                "expected_answer": expected_answer,
                "contexts": contexts,
                "qa_score": qa_eval_result.get("score", 0),
                "context_relevance_score": context_score_num,
                "context_text": context_text[:200] + "..." if len(context_text) > 200 else context_text
            })
            
        except Exception as e:
            print(f"    ❌ Error: {str(e)}")
            results.append({
                "question": question,
                "generated_answer": f"Error: {str(e)}",
                "expected_answer": expected_answer,
                "contexts": [],
                "qa_score": 0,
                "context_relevance_score": 0,
                "context_text": ""
            })
    
    total_time = time.time() - start_time
    print(f"  ✅ Completed in {total_time:.1f} seconds")
    
    return results, total_time

In [ ]:
# Run  evaluation on all retrievers
def run_langchain_evaluation():
    
    print("🚀 Activity 1: Retriever Evaluation")
    print("=" * 60)
    
    # Define retrievers to evaluate
    retrievers_to_evaluate = {
        "Naive": (naive_retriever, naive_retrieval_chain),
        "BM25": (bm25_retriever, bm25_retrieval_chain),
        "Compression": (compression_retriever, contextual_compression_retrieval_chain),
        "Multi-Query": (multi_query_retriever, multi_query_retrieval_chain),
        "Parent-Document": (parent_document_retriever, parent_document_retrieval_chain),
        "Ensemble": (ensemble_retriever, ensemble_retrieval_chain)
    }
    
    evaluation_results = {}
    
    # Evaluate each retriever
    for retriever_name, (retriever, chain) in retrievers_to_evaluate.items():
        print(f"\n{'='*20} {retriever_name} {'='*20}")
        
        results, time_taken = evaluate_retriever_langchain(retriever_name, retriever, chain)
        
        evaluation_results[retriever_name] = {
            "results": results,
            "time_taken": time_taken
        }
    
    return evaluation_results

# Run the evaluation
evaluation_results = run_langchain_evaluation()

🚀 Activity 1: LangChain Retriever Evaluation

==================== Naive ====================

🔍 Evaluating Naive with LangChain...
  Q1: What is the most common project domain?...


/var/folders/3y/wtv8pmq51xl97283kkkq1p300000gn/T/ipykernel_64208/2961211990.py:21: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(question)


  Q2: Which projects received the highest scores?...
  Q3: What did judges say about fintech projects?...
  Q4: Which projects use medical imaging?...
  Q5: What are the BioForge projects about?...
  Q6: Which projects scored above 90?...
  Q7: What feedback did judges give about UI design?...
  Q8: Which projects focus on security applications?...
  ✅ Completed in 21.8 seconds

==================== BM25 ====================

🔍 Evaluating BM25 with LangChain...
  Q1: What is the most common project domain?...
  Q2: Which projects received the highest scores?...
  Q3: What did judges say about fintech projects?...
  Q4: Which projects use medical imaging?...
  Q5: What are the BioForge projects about?...
  Q6: Which projects scored above 90?...
  Q7: What feedback did judges give about UI design?...
  Q8: Which projects focus on security applications?...
  ✅ Completed in 16.5 seconds

==================== Compression ====================

🔍 Evaluating Compression with LangChain...
  Q1:

In [60]:
# Analyze LangChain evaluation results
def analyze_langchain_results(evaluation_results):
    """Analyze and compare results from LangChain evaluation"""
    
    print("\n EVALUATION RESULTS")
    print("=" * 60)
    
    comparison_data = []
    
    for retriever_name, data in evaluation_results.items():
        results = data["results"]
        
        # Calculate average scores
        qa_scores = [r["qa_score"] for r in results if r["qa_score"] > 0]
        context_scores = [r["context_relevance_score"] for r in results if r["context_relevance_score"] > 0]
        
        avg_qa_score = sum(qa_scores) / len(qa_scores) if qa_scores else 0
        avg_context_score = sum(context_scores) / len(context_scores) if context_scores else 0
        
        comparison_data.append({
            "Retriever": retriever_name,
            "Avg_QA_Score": avg_qa_score,
            "Avg_Context_Score": avg_context_score,
            "Overall_Score": (avg_qa_score + avg_context_score) / 2,
            "Latency": data["time_taken"],
            "Success_Rate": len([r for r in results if r["qa_score"] > 0]) / len(results)
        })
    
    # Print performance comparison table
    print("\nPERFORMANCE COMPARISON:")
    print("-" * 80)
    print(f"{'Retriever':<15} {'QA Score':<10} {'Context':<10} {'Overall':<10} {'Latency':<10} {'Success':<10}")
    print("-" * 80)
    
    for data in comparison_data:
        print(f"{data['Retriever']:<15} {data['Avg_QA_Score']:<10.2f} {data['Avg_Context_Score']:<10.2f} "
              f"{data['Overall_Score']:<10.2f} {data['Latency']:<10.1f} {data['Success_Rate']:<10.2f}")
    
    # Find best performers
    print("\n🏆 BEST PERFORMERS:")
    best_qa = max(comparison_data, key=lambda x: x["Avg_QA_Score"])
    best_context = max(comparison_data, key=lambda x: x["Avg_Context_Score"])
    best_overall = max(comparison_data, key=lambda x: x["Overall_Score"])
    fastest = min(comparison_data, key=lambda x: x["Latency"])
    
    print(f"  Best QA Score: {best_qa['Retriever']} ({best_qa['Avg_QA_Score']:.2f})")
    print(f"  Best Context Score: {best_context['Retriever']} ({best_context['Avg_Context_Score']:.2f})")
    print(f"  Best Overall: {best_overall['Retriever']} ({best_overall['Overall_Score']:.2f})")
    print(f"  Fastest: {fastest['Retriever']} ({fastest['Latency']:.1f}s)")
    
    return comparison_data

# Analyze results
results_df = analyze_langchain_results(evaluation_results)


 EVALUATION RESULTS

PERFORMANCE COMPARISON:
--------------------------------------------------------------------------------
Retriever       QA Score   Context    Overall    Latency    Success   
--------------------------------------------------------------------------------
Naive           1.00       12.88      6.94       21.8       0.25      
BM25            1.00       1.38       1.19       16.5       0.25      
Compression     1.00       1.88       1.44       22.1       0.25      
Multi-Query     1.00       2.00       1.50       46.4       0.38      
Parent-Document 1.00       1.50       1.25       21.3       0.38      
Ensemble        1.00       2.12       1.56       58.3       0.12      

🏆 BEST PERFORMERS:
  Best QA Score: Naive (1.00)
  Best Context Score: Naive (12.88)
  Best Overall: Naive (6.94)
  Fastest: BM25 (16.5s)


Based on the comprehensive LangChain evaluation of retriever methods, the Naive retriever demonstrates superior overall performance with a combined score of 6.94, achieving the highest QA score of 1.00 and context relevance score of 12.88.

From a performance perspective, the Naive retriever excels in retrieving contextually relevant information for our project dataset, effectively handling diverse query types from domain-specific questions to technical implementation details. The BM25 retriever offers optimal latency at 16.5 seconds, making it suitable for real-time applications.

Considering cost, latency, and performance trade-offs, the Naive retriever provides the best balance for this particular dataset, leveraging LangChain's evaluation framework to ensure both answer quality and context relevance while maintaining computational efficiency.